# 01_preprocesamiento.ipynb
Preparación del dataset analítico (between-subject)

**Correcciones respecto a versión anterior:**
- Agregado filtro por chequeo de atención (exclusión a nivel de bloque)
- Agregado chi² de exclusión diferencial entre condiciones
- Agregado test de equivalencia entre excluidos y retenidos (Mann-Whitney)
- Agregado centrado de variables `politica` y `nivel_se`

In [1]:
import pandas as pd
import numpy as np
from scipy import stats

df = pd.read_csv("df_long0.csv")
print("Shape original:", df.shape)
df.head()

Shape original: (3485, 32)


,ID_Sujeto,Origen_Form,Identidad,Dilema,Orden_1,Bloque,Orden_2,Respuesta,Mantiene,SDO_Score,...,Promedio_Gap_0.0,Promedio_Gap_1000.0,Promedio_Gap_1200.0,Promedio_Gap_2000.0,Promedio_Gap_2400.0,Promedio_CON,Promedio_SIN,Promedio_DIST,Delta_Mantiene,Delta_base
0,Suj_001,Respuestas de formulario 1,ABC,Bloque_CON,1,Bloque_1,1,Opción 2,0,2.1,...,0.666667,0.666667,0.666667,0.333333,0.0,0.166667,0.666667,0.666667,-0.5,0.0
1,Suj_001,Respuestas de formulario 1,AEF,Bloque_CON,1,Bloque_5,2,Opción 2,0,2.1,...,0.666667,0.666667,0.666667,0.333333,0.0,0.166667,0.666667,0.666667,-0.5,0.0
2,Suj_001,Respuestas de formulario 1,AJK,Bloque_CON,1,Bloque_7,3,Opción 2,0,2.1,...,0.666667,0.666667,0.666667,0.333333,0.0,0.166667,0.666667,0.666667,-0.5,0.0
3,Suj_001,Respuestas de formulario 1,-,Atencion,1,Bloque_0,1,Opción 3,0,2.1,...,0.666667,0.666667,0.666667,0.333333,0.0,0.166667,0.666667,0.666667,-0.5,0.0
4,Suj_001,Respuestas de formulario 1,AGH,Bloque_CON,1,Bloque_3,4,Opción 2,0,2.1,...,0.666667,0.666667,0.666667,0.333333,0.0,0.166667,0.666667,0.666667,-0.5,0.0


## 1. Filtro por chequeo de atención

Cada bloque contiene 1 escenario de atención intercalado (Dilema == 'Atencion').
Respuesta correcta: 'Opción 3'.
Se excluyen las observaciones del bloque donde el participante falló el chequeo.
Criterio aplicado a nivel de bloque, no de sujeto.

In [2]:
# Identificar chequeos de atención y marcar bloques fallados
atencion = df[df['Dilema'] == 'Atencion'][['ID_Sujeto', 'Orden_1', 'Respuesta']].copy()
atencion['paso'] = (atencion['Respuesta'] == 'Opción 3').astype(int)

excluir = atencion[atencion['paso'] == 0][['ID_Sujeto', 'Orden_1']].copy()
excluir['excluir'] = 1

print(f"Total chequeos de atención: {len(atencion)}")
print(f"Chequeos fallados (bloques a excluir): {len(excluir)}")
print()
print("Fallos por condición (Orden_1):")
print(atencion.groupby('Orden_1')['paso'].apply(lambda x: (x==0).sum()))

Total chequeos de atención: 498
Chequeos fallados (bloques a excluir): 83

Fallos por condición (Orden_1):
Orden_1
1    33
2    27
3    23
Name: paso, dtype: int64


## 2. Verificación: tasa de exclusión diferencial entre condiciones

In [3]:
# Tasa de exclusión por condición (tratamiento en primer bloque)
df_exp_todos = df[df['Dilema'] != 'Atencion'].copy()

# Mapear condición a Orden_1=1
cond_map = df_exp_todos[df_exp_todos['Orden_1'] == 1][['ID_Sujeto', 'Dilema']].drop_duplicates()

atencion_merged = atencion.merge(cond_map, on='ID_Sujeto', how='left')
atencion_merged['Condicion'] = atencion_merged['Dilema']

# Solo Orden_1 == 1 para la tabla de exclusión
aten_first = atencion_merged[atencion_merged['Orden_1'] == 1]
tabla_excl = aten_first.groupby('Condicion')['paso'].agg(
    total='count',
    aprobados='sum'
).assign(fallidos=lambda x: x['total'] - x['aprobados'],
         tasa_fallo=lambda x: (x['total'] - x['aprobados']) / x['total'])
print(tabla_excl)
print()

# Chi-cuadrado de exclusión diferencial
contingencia = tabla_excl[['aprobados', 'fallidos']].values
chi2, p_chi, dof, _ = stats.chi2_contingency(contingencia)
print(f"Chi-cuadrado de exclusión diferencial: χ²({dof}) = {chi2:.2f}, p = {p_chi:.3f}")
print("→ Si p > .05: la tasa de exclusión no difiere significativamente entre condiciones.")

            total  aprobados  fallidos  tasa_fallo
Condicion                                         
Bloque_CON     49         36        13    0.265306
Bloque_SIN     57         47        10    0.175439
Dist           60         50        10    0.166667

Chi-cuadrado de exclusión diferencial: χ²(2) = 1.95, p = 0.378
→ Si p > .05: la tasa de exclusión no difiere significativamente entre condiciones.


## 3. Equivalencia entre excluidos y retenidos en condición CON

Verificar que los participantes excluidos de CON (primer bloque) no difieran de los retenidos
en variables de control → descartar sesgo sistemático por exclusión diferencial.

In [4]:
# Participantes en CON como primer bloque
df_con_first = df_exp_todos[
    (df_exp_todos['Orden_1'] == 1) & (df_exp_todos['Dilema'] == 'Bloque_CON')
][['ID_Sujeto', 'NDC_Score', 'SDO_Score', 'politica', 'nivel_se']].drop_duplicates('ID_Sujeto')

# Marcar excluidos
excluir_con = excluir[excluir['Orden_1'] == 1].merge(cond_map, on='ID_Sujeto')
excluir_con_ids = excluir_con[excluir_con['Dilema'] == 'Bloque_CON']['ID_Sujeto'].unique()

df_con_first['excluido'] = df_con_first['ID_Sujeto'].isin(excluir_con_ids)

excl = df_con_first[df_con_first['excluido']]
ret  = df_con_first[~df_con_first['excluido']]

print(f"CON primer bloque: excluidos n={len(excl)}, retenidos n={len(ret)}")
print()
for var in ['NDC_Score', 'SDO_Score', 'politica', 'nivel_se']:
    u, p = stats.mannwhitneyu(excl[var].dropna(), ret[var].dropna(), alternative='two-sided')
    print(f"  {var:15s}: U={u:.1f}, p={p:.3f}")
print("→ Si todas las p > .05: no hay sesgo sistemático por exclusión.")

CON primer bloque: excluidos n=13, retenidos n=36

  NDC_Score      : U=204.5, p=0.509
  SDO_Score      : U=245.5, p=0.803
  politica       : U=301.5, p=0.123
  nivel_se       : U=249.5, p=0.728
→ Si todas las p > .05: no hay sesgo sistemático por exclusión.


## 4. Aplicar filtro y construir dataset between-subject

In [5]:
# Remover filas de atención y aplicar filtro de bloque
df_exp = df[df['Dilema'] != 'Atencion'].copy()
df_filt = df_exp.merge(excluir, on=['ID_Sujeto', 'Orden_1'], how='left')
df_filt = df_filt[df_filt['excluir'].isna()].drop(columns='excluir')

# Análisis between-subject: solo primer bloque recibido
df_between = df_filt[df_filt['Orden_1'] == 1].copy()

print(f"N sujetos: {df_between['ID_Sujeto'].nunique()}")
print(f"N observaciones: {len(df_between)}")
print()
print("Distribución por condición:")
print(df_between.groupby('Dilema')['ID_Sujeto'].nunique())

N sujetos: 133
N observaciones: 798

Distribución por condición:
Dilema
Bloque_CON    36
Bloque_SIN    47
Dist          50
Name: ID_Sujeto, dtype: int64


## 5. Crear variables analíticas

In [6]:
# Tratamiento (referencia = Dist)
df_between['Tratamiento'] = pd.Categorical(
    df_between['Dilema'],
    categories=['Dist', 'Bloque_SIN', 'Bloque_CON']
)

# Variable dependiente
df_between['Mantiene_bin'] = df_between['Mantiene'].astype(int)

# NDC centrada
df_between['NDC_c'] = df_between['NDC_Score'] - df_between['NDC_Score'].mean()

# SDO centrada
df_between['SDO_c'] = df_between['SDO_Score'] - df_between['SDO_Score'].mean()

# Política centrada (CORRECCIÓN: antes entraba sin centrar)
df_between['pol_c'] = df_between['politica'] - df_between['politica'].mean()

# Nivel SE centrado (CORRECCIÓN: antes entraba sin centrar)
df_between['nse_c'] = df_between['nivel_se'] - df_between['nivel_se'].mean()

# Género dummy (mujer = 1)
df_between['Gen_mujer'] = (df_between['Genero'] == 'Mujer').astype(int)

# Gap
df_between['Gap'] = df_between['Gap_Size']

# Dummy Gap = 0
df_between['Gap0'] = (df_between['Gap'] == 0).astype(int)

# Gap positivo centrado en media de gaps > 0, expresado en miles de ARS
gap_pos_mean = df_between.loc[df_between['Gap'] > 0, 'Gap'].mean()
print(f"Media de gaps positivos: {gap_pos_mean:.1f} ARS")
df_between['Gap_pos'] = np.where(
    df_between['Gap'] > 0,
    (df_between['Gap'] - gap_pos_mean) / 1000,
    np.nan
)

print("\nVariables creadas:", [c for c in df_between.columns if c not in df.columns])

Media de gaps positivos: 1650.0 ARS

Variables creadas: ['Tratamiento', 'Mantiene_bin', 'NDC_c', 'SDO_c', 'pol_c', 'nse_c', 'Gen_mujer', 'Gap', 'Gap0', 'Gap_pos']


## 6. Tabla de contrabalanceo (Tabla S1)

Distribución de participantes por secuencia de presentación de bloques.

In [7]:
# Reconstruir secuencia completa por sujeto
df_seq = df_exp[df_exp['Dilema'] != 'Atencion'].copy()
seq = df_seq.groupby('ID_Sujeto').apply(
    lambda x: x.drop_duplicates('Orden_1').sort_values('Orden_1')['Dilema'].tolist()
).reset_index()
seq.columns = ['ID_Sujeto', 'secuencia']
seq['secuencia_str'] = seq['secuencia'].apply(lambda x: ' → '.join(
    [s.replace('Bloque_','').replace('Dist','DIST') for s in x]
))

tabla_s1 = seq['secuencia_str'].value_counts().reset_index()
tabla_s1.columns = ['Secuencia de presentación', 'n']
tabla_s1 = tabla_s1.sort_values('Secuencia de presentación')
tabla_s1.loc[len(tabla_s1)] = ['Total', tabla_s1['n'].sum()]
print("Tabla S1 — Distribución de secuencias")
print(tabla_s1.to_string(index=False))

Tabla S1 — Distribución de secuencias
Secuencia de presentación   n
         CON → DIST → SIN  26
         CON → SIN → DIST  23
         DIST → CON → SIN  31
         DIST → SIN → CON  29
         SIN → CON → DIST  28
         SIN → DIST → CON  29
                    Total 166


## 7. Equivalencia basal por condición recibida primero (Tablas S2 y S3)

In [8]:
# Una fila por sujeto con su condición en primer lugar
df_suj = df_between[['ID_Sujeto','Dilema','NDC_Score','SDO_Score',
                      'politica','nivel_se','Genero']].drop_duplicates('ID_Sujeto')
df_suj = df_suj.rename(columns={'Dilema':'Condicion_1'})

print("Tabla S2 — Equivalencia en variables de control por condición recibida primero")
print()
for var, label in [('NDC_Score','NDC'), ('SDO_Score','SDO'),
                   ('politica','Autoposicionamiento político'), ('nivel_se','Nivel SE')]:
    grupos = [df_suj[df_suj['Condicion_1']==c][var].dropna()
              for c in ['Bloque_CON','Bloque_SIN','Dist']]
    H, p = stats.kruskal(*grupos)
    ms = [f"{g.mean():.2f} ({g.std():.2f})" for g in grupos]
    print(f"  {label:35s}: CON={ms[0]}, SIN={ms[1]}, DIST={ms[2]}, H={H:.2f}, p={p:.3f}")

print()
print("Tabla S3 — Distribución de género por condición recibida primero")
crosstab = pd.crosstab(df_suj['Condicion_1'], df_suj['Genero'])
print(crosstab)
chi2_g, p_g, _, _ = stats.chi2_contingency(crosstab.values)
print(f"Chi-cuadrado: χ²({crosstab.shape[0]*crosstab.shape[1]-crosstab.shape[0]-crosstab.shape[1]+1}) = {chi2_g:.2f}, p = {p_g:.3f}")

Tabla S2 — Equivalencia en variables de control por condición recibida primero

  NDC                                : CON=3.72 (0.64), SIN=3.85 (0.64), DIST=4.01 (0.71), H=4.53, p=0.104
  SDO                                : CON=2.16 (0.73), SIN=2.13 (0.79), DIST=2.07 (0.72), H=0.40, p=0.820
  Autoposicionamiento político       : CON=3.64 (1.55), SIN=3.89 (1.37), DIST=3.62 (1.31), H=1.09, p=0.581
  Nivel SE                           : CON=5.00 (1.49), SIN=5.70 (1.44), DIST=5.86 (1.40), H=7.09, p=0.029

Tabla S3 — Distribución de género por condición recibida primero
Genero       Hombre  Mujer  No binario  Otro
Condicion_1                                 
Bloque_CON        9     27           0     0
Bloque_SIN       14     31           1     1
Dist             17     31           2     0
Chi-cuadrado: χ²(6) = 4.42, p = 0.619


## 8. Guardar dataset analítico

In [9]:
df_between.to_csv("dataset_between.csv", index=False)
print(f"Dataset guardado: dataset_between.csv")
print(f"Shape: {df_between.shape}")
print(f"N sujetos: {df_between['ID_Sujeto'].nunique()}")

Dataset guardado: dataset_between.csv
Shape: (798, 42)
N sujetos: 133
